# Project8 — L2 数据探查

样本：`sh600000` 20250102

In [ ]:
import sys
sys.path.insert(0, '/home/hysheng/project8')
sys.path.insert(0, '/home/hysheng')

import numpy as np
import pandas as pd
from data.loader import load_daily

order, tick, trade = load_daily('20250102', 'sh600000')
from_unix = lambda t: pd.to_datetime(t, unit='s', utc=True).tz_convert('Asia/Shanghai')
print("order:", order.shape, "  tick:", tick.shape, "  trade:", trade.shape)

## 1. order 表

In [ ]:
print("字段/dtype:")
print(order.dtypes.to_string())
print()
print("OrderType 分布 (4=限价, 5=撤单指令, 0=市价):")
print(order['OrderType'].value_counts().to_string())
print()
print("BSFlag 分布 (0=买, 1=卖):")
print(order['BSFlag'].value_counts().to_string())
print()
t_ex = order['OrderTime'].iloc[10]
print(f"时间戳精度示例: {t_ex}  小数部分={t_ex - int(t_ex):.3f}s (毫秒级)")
print(f"时间范围: {from_unix(order['OrderTime'].min())} ~ {from_unix(order['OrderTime'].max())}")
print()
print("缺失值:", order.isnull().sum().to_string())

## 2. tick 表

In [ ]:
print("字段数:", len(tick.columns))
print("字段:", list(tick.columns))
print()
print(f"time 范围: {from_unix(tick['time'].min())} ~ {from_unix(tick['time'].max())}")
dt = tick['time'].diff().dropna()
print(f"tick 间隔(秒): mean={dt.mean():.2f}, std={dt.std():.2f}, min={dt.min():.2f}, max={dt.max():.0f}")
print()
# 连续竞价时段 tick 数
local_sec = (tick['time'] + 28800) % 86400
cont_mask = (local_sec >= 9*3600+30*60) & (local_sec < 15*3600)
print(f"连续竞价 09:30-15:00 tick 数: {cont_mask.sum()} / {len(tick)}")
print()
# 盘口零值（涨跌停检测）
ask_empty = (tick['akp1'] == 0) & (tick['bdp1'] > 0)
bid_empty = (tick['bdp1'] == 0) & (tick['akp1'] > 0)
print(f"涨停 tick 数 (ask1==0): {ask_empty.sum()}")
print(f"跌停 tick 数 (bid1==0): {bid_empty.sum()}")
print()
print("关键列缺失值:", tick[['time','bdp1','akp1','bdv1','akv1']].isnull().sum().to_string())

## 3. trade 表

In [ ]:
print("字段/dtype:")
print(trade.dtypes.to_string())
print()
print("TradeCode 分布 (0=成交, 4=撤单/深交所):")
print(trade['TradeCode'].value_counts().to_string())
print()
print("BSFlag 分布 (0=主买Taker, 1=主卖Taker):")
print(trade['BSFlag'].value_counts().to_string())
print()
real = trade[trade['TradeCode']==0]
print(f"真实成交: {len(real)} 笔")
print(f"时间范围: {from_unix(real['TradeTime'].min())} ~ {from_unix(real['TradeTime'].max())}")
print()
print("缺失值:", trade.isnull().sum().to_string())

## 4. 时间戳对齐验证（无前视）

In [ ]:
from data.snapshot_aligner import align_orders_to_tick

merged = align_orders_to_tick(order, tick)
valid = merged.dropna(subset=['time'])
lookahead = (valid['time'] > valid['OrderTime']).sum()
print(f"前视违规数: {lookahead}  (预期=0)")
print(f"成功匹配率: {len(valid)/len(order)*100:.1f}%")

## 5. 涨跌停 Regime 标注规则说明

- **涨停**：`tick.akp1 == 0` 且 `tick.bdp1 > 0`（卖档为空，只有买方排队）
- **跌停**：`tick.bdp1 == 0` 且 `tick.akp1 > 0`（买档为空，只有卖方排队）
- 处理方式：因子计算时剔除涨跌停 tick，或单独打 `regime=limit_up/limit_down` 标签后在因子层面分层评估。

## 6. 关键字段汇总

| 字段 | 表 | 类型 | 说明 |
|------|----|------|------|
| `OrderTime` | order | float64 UNIX秒 | 毫秒精度（小数位）|
| `sysid` | order/trade | float64 | 委托编号，跨表匹配主键 |
| `OrderType` | order | float64 | 4=限价，5=上交所撤单指令，0=市价 |
| `BSFlag` | order/trade | float64 | 0=买，1=卖 |
| `TradeCode` | trade | int64 | 0=成交，4=深交所撤单 |
| `BuyOrderID/SellOrderID` | trade | float64 | 买卖方委托号，匹配 order.sysid |
| `time` | tick | float64 UNIX秒 | 快照发布时刻，约4秒间隔 |
| `bdp1~10/bdv1~10` | tick | float64 | 买档价/量（10档）|
| `akp1~10/akv1~10` | tick | float64 | 卖档价/量（10档）|
| `totalVolumeDiff/totalAmtDiff` | tick | float64 | tick间增量，用于 running VWAP |